# Multi-Head Ablation: Quantifying Circuit Redundancy
*Ablates groups of attention heads simultaneously to measure the redundancy factor.*

In [ ]:
!pip install -q torch transformers scikit-learn matplotlib seaborn

In [ ]:
import torch, numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from transformers import AutoTokenizer, AutoModelForCausalLM
import json, re
from tqdm import tqdm

SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

In [ ]:
DATASET_PATH = "typosquat_dataset_full/typosquat_tool_calls.jsonl"
df_all = pd.DataFrame([json.loads(line) for line in open(DATASET_PATH)])
df_adv = df_all[df_all['is_adversarial'] == True].sample(n=300, random_state=SEED).copy()
print(f"Sampled {len(df_adv)} adversarial entries")

texts, labels = [], []
for _, row in df_adv.iterrows():
    texts.append(row['clean_command']); labels.append(0)
    texts.append(row['typo_command']); labels.append(1)

In [ ]:
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float16, device_map="auto")
model.eval()

num_layers = model.config.num_hidden_layers
num_heads = model.config.num_attention_heads
hidden_size = model.config.hidden_size
head_dim = hidden_size // num_heads
print(f"Layers: {num_layers}, Heads: {num_heads}, Head dim: {head_dim}")

In [ ]:
def extract_features(model, tokenizer, texts, batch_size=16):
    feats = []
    for i in tqdm(range(0, len(texts), batch_size), desc="Extracting baseline"):
        batch = tokenizer(texts[i:i+batch_size], padding=True, truncation=True, max_length=64, return_tensors="pt").to(DEVICE)
        with torch.no_grad():
            outs = model(**batch, output_hidden_states=True)
            feats.append(outs.hidden_states[-1].mean(dim=1).float().cpu().numpy())
    return np.vstack(feats)

X_base = extract_features(model, tokenizer, texts)
y = np.array(labels)
probe_base = LogisticRegression(max_iter=1000, random_state=SEED).fit(X_base, y)
auc_base = roc_auc_score(y, probe_base.predict_proba(X_base)[:, 1])
print(f"Baseline AUC: {auc_base:.4f}")

In [ ]:
def extract_with_head_ablation(ablated_heads):
    hooks = []
    def make_hook(layer_idx, head_idx):
        def hook_fn(module, input, output):
            bsz, seq, hidden = output.shape
            output_reshaped = output.view(bsz, seq, num_heads, head_dim)
            output_reshaped[:, :, head_idx, :] = 0.0
            return output_reshaped.view(bsz, seq, hidden)
        return hook_fn
    for layer_idx, head_idx in ablated_heads:
        layer = model.model.layers[layer_idx]
        hook = layer.self_attn.o_proj.register_forward_hook(make_hook(layer_idx, head_idx))
        hooks.append(hook)
    feats = []
    for i in tqdm(range(0, len(texts), 8), desc="Extracting with ablation"):
        batch = tokenizer(texts[i:i+8], padding=True, truncation=True, max_length=64, return_tensors="pt").to(DEVICE)
        with torch.no_grad():
            outs = model(**batch, output_hidden_states=True)
            feats.append(outs.hidden_states[-1].mean(dim=1).float().cpu().numpy())
    for hook in hooks: hook.remove()
    return np.vstack(feats)

target_layer = 1  # From prior head ablation experiment

k_values = [2, 4, 6, 8, 10, 12]
results = []
for k in k_values:
    print(f"\n=== Ablating {k} heads simultaneously ===")
    heads_to_ablate = [(target_layer, h) for h in range(k)]
    X_abl = extract_with_head_ablation(heads_to_ablate)
    probe_abl = LogisticRegression(max_iter=1000, random_state=SEED).fit(X_abl, y)
    auc_abl = roc_auc_score(y, probe_abl.predict_proba(X_abl)[:, 1])
    drop = auc_base - auc_abl
    results.append({'k': k, 'auc': auc_abl, 'drop': drop})
    print(f"k={k:2d}: AUC={auc_abl:.4f} (Δ={drop:+.4f})")

res_df = pd.DataFrame(results)

In [ ]:
plt.figure(figsize=(8,5))
plt.plot(res_df['k'], res_df['auc'], 'o-', linewidth=2, markersize=8)
plt.axhline(y=0.5, ls='--', c='gray', label='Chance (0.5)')
plt.axhline(y=0.9, ls=':', c='orange', label='High-performance threshold (0.9)')
plt.xlabel('Number of Simultaneously Ablated Heads')
plt.ylabel('Probe AUC')
plt.title('Redundancy Curve: Typosquat Detection vs. Head Ablation')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('redundancy_curve.png', dpi=150)
plt.show()

threshold = 0.90
below_thresh = res_df[res_df['auc'] < threshold]
if len(below_thresh) > 0:
    redundancy_factor = below_thresh['k'].min()
    print(f"\n✓ Redundancy factor: {redundancy_factor} heads must be ablated to drop AUC below {threshold}")
else:
    print(f"\n⚠ AUC remains ≥ {threshold} even after ablating all {num_heads} heads")
    print("  → Circuit is maximally redundant; consider layer-wise or MLP ablation next.")